In [12]:
import torch

In [13]:
from model.transformer import TransformerBlock
from model.cross import XBlock
from model.embedding import StableEmbedding
from model.dag_head import OutputDAG

In [14]:
vocab_size = 10
pad_idx = 0
max_vertices = 12 # decoder
max_seq_len = 6 # encoder

In [38]:
factor = 2

In [15]:
batch_size = 2

In [16]:
encoder_tokens = []
encoder_lens = []
for i in range(batch_size):
    curr_len = torch.randint(1, max_seq_len, (1,)).item()
    encoder_lens.append(curr_len)
    encoder_tokens.append(torch.randint(1, vocab_size, (curr_len,)))

In [17]:
encoder_lens = torch.tensor(encoder_lens)

In [18]:
encoder_tokens = torch.nested.nested_tensor(encoder_tokens)

In [19]:
encoder_tokens = torch.nested.to_padded_tensor(encoder_tokens, pad_idx, (batch_size, max_seq_len))

In [20]:
decoder_tokens = torch.arange(0, max_vertices).unsqueeze(0).repeat(batch_size, 1)

In [39]:
vertex_lens = encoder_lens * factor

In [40]:
encoder_lens, vertex_lens # (tensor([5, 3]), tensor([10,  6]))

(tensor([5, 3]), tensor([10,  6]))

In [35]:
encoder_tokens

tensor([[4, 9, 2, 3, 8, 0],
        [5, 1, 3, 0, 0, 0]])

In [36]:
encoder_is_pad = encoder_tokens == pad_idx

In [37]:
encoder_is_pad

tensor([[False, False, False, False, False,  True],
        [False, False, False,  True,  True,  True]])

In [43]:
decoder_is_pad = encoder_is_pad.repeat_interleave(factor, dim=1)

In [44]:
decoder_is_pad

tensor([[False, False, False, False, False, False, False, False, False, False,
          True,  True],
        [False, False, False, False, False, False,  True,  True,  True,  True,
          True,  True]])

In [45]:
datafile = "data/processed.pt"

In [46]:
result = torch.load(datafile)

In [48]:
ens, zhs = result

In [50]:
# sample ens based on batch_size
ens = ens[0:batch_size]

In [52]:
# count number of 65000 values per row in ens
ens_lens = torch.sum(ens == 65000, dim=1)

In [65]:
torch.sum(ens != 65000, dim=1)

tensor([12, 30])

In [57]:
zhs.shape

torch.Size([999340, 513])

In [58]:
total_samples, length = zhs.shape

In [61]:
ens_lens = length - ens_lens

In [66]:
ens_lens

tensor([12, 30])

In [62]:
vector_lens = ens_lens * factor

In [63]:
encder_is_pad = ens == 65000

In [64]:
ens_mask = encder_is_pad.repeat_interleave(factor, dim=1)

In [74]:
from utils.data import process_data

In [75]:
pl, _, plm, pm = process_data(ens, 65000, factor)

ValueError: not enough values to unpack (expected 3, got 2)

In [76]:
# pm should be the same as ens_mask
assert torch.all(pm == ens_mask)